# 覓聲 MeetSum — 錯字校正 LoRA 微調（國語＋英文）

**做什麼**：讓 Qwen3.8-27B 更會抓語音辨識造成的同音／近音錯字。原本的 `qwen3.8:27b` 完全不動，
微調結果另存成 Ollama 模型 `qwen3.8-27b-typo`，評分比原版好才會接進系統。

**資料**：AISHELL-4（國語真實會議）、AMI（英文真實會議）用專案自己的 ASR 辨識，
和人工逐字稿比對出的同音錯字當標準答案；另加規則式同音字注入。沒有用任何大型語言模型產生答案。

**用法**：執行階段選 GPU（96 GB）→ 從上到下依序執行。每一步的結果都存在雲端硬碟 `MeetSum_finetune/`，
Colab 斷線後重新連線、從頭再按一次即可，做過的步驟會自動跳過。

⚠️ 全程約 6–8 小時，請保持這個分頁開著、電腦不要睡眠。

In [21]:
# 0. 設定與檢查
import os, subprocess, sys, shutil, json, glob, time
BASE_TAG = 'qwen3.8:27b'                               # 正式系統用的 Ollama 模型
FT_TAG = 'qwen3.8-27b-typo'                            # 微調後的 Ollama 模型名稱
BASE_HF = 'Qwen/Qwen3.8-27B'                           # 訓練用的原始權重
BASE_HF_DIR = '/content/hf/Qwen3.8-27B'
SCALE = 1.0                                            # 會議數量倍率；想先試跑一小輪改成 0.1

try:
    from google.colab import drive
    if not os.path.isdir('/content/drive/MyDrive'):
        drive.mount('/content/drive')
    WORK = '/content/drive/MyDrive/MeetSum_finetune'   # 結果存雲端硬碟，斷線不會不見
except Exception as e:
    print('⚠️ 無法掛載雲端硬碟（', type(e).__name__, e, '）→ 結果先存在 Colab 本機，斷線會消失')
    WORK = '/content/MeetSum_finetune'
for d in ['data', 'results', 'run1', 'export', 'logs', 'code_override']:
    os.makedirs(f'{WORK}/{d}', exist_ok=True)
print('結果資料夾：', WORK)
DATA = f'{WORK}/data2'      # v2：單一麥克風聲道（v1 的 data/ 是 8 聲道平均，辨識太差，不使用）
os.makedirs(DATA, exist_ok=True)
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'], capture_output=True, text=True).stdout)
print('磁碟剩餘', shutil.disk_usage('/content').free // 2**30, 'GB')
GPU_GB = int(subprocess.run(['nvidia-smi', '--query-gpu=memory.total', '--format=csv,noheader,nounits'],
                            capture_output=True, text=True).stdout.split()[0]) // 1024

def run(cmd, log_name=None, env=None):
    # 執行指令，即時印出，並另存 log。執行前先套用雲端硬碟 code_override/ 裡的修正版程式（遠端修 bug 用）
    for f in glob.glob(f'{WORK}/code_override/*.py'):
        shutil.copy(f, f'{FT}/{os.path.basename(f)}')
        print('套用修正版：', os.path.basename(f))
    print('$', ' '.join(cmd), flush=True)
    logf = open(f'{WORK}/logs/{log_name}.log', 'a', encoding='utf-8') if log_name else None
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
                         env={**os.environ, **(env or {})})
    try:
        for line in p.stdout:
            if 'it/s]' in line or 's/it]' in line:
                print(line.rstrip()[-160:], end='\r', flush=True)
            else:
                print(line, end='', flush=True)
            if logf:
                logf.write(line); logf.flush()
        p.wait()
    except KeyboardInterrupt:      # 按「中斷」時連子程序一起停掉，不留在背景占用 GPU
        p.kill(); p.wait()
        raise
    if p.returncode != 0:
        raise RuntimeError(f'失敗（exit {p.returncode}），詳見 {WORK}/logs/{log_name}.log')

def done(path):
    return os.path.exists(path)

# 停掉上一次中斷後可能殘留在背景的子程序
subprocess.run(['pkill', '-f', 'ft_typo/(build_asr_data|build_sft|eval_typo|train_lora|export_ollama).py'])

# 微調程式直接內嵌在 notebook 裡，寫到 /content/ft_typo（不依賴雲端硬碟同步）
FT = '/content/ft_typo'
os.makedirs(FT, exist_ok=True)
EMBED = {'ft_common.py': '"""錯字校正微調：共用工具（正規化、對齊、標註、組 prompt、評分）。\n\n標準答案一律來自「人工逐字稿 vs 專案 ASR 辨識結果」的差異，或程式規則注入的同音字，\n不使用任何大型語言模型產生的答案。\n"""\nfrom __future__ import annotations\n\nimport difflib\nimport os\nimport random\nimport re\nimport sys\nfrom functools import lru_cache\nfrom pathlib import Path\n\n# ---------------------------------------------------------------- 專案程式（prompt、解析器都用正式版本）\n\ndef project_backend() -> Path:\n    env = os.environ.get("PROJECT_BACKEND")\n    if env:\n        return Path(env)\n    here = Path(__file__).resolve()\n    for p in [here.parents[2] / "backend", Path("/content/Taiwanese_Meeting_Assistant/backend")]:\n        if (p / "app" / "services" / "typo_service.py").exists():\n            return p\n    raise RuntimeError("找不到專案 backend，請設定環境變數 PROJECT_BACKEND")\n\n\ndef typo_service():\n    b = str(project_backend())\n    if b not in sys.path:\n        sys.path.insert(0, b)\n    from app.services import typo_service as ts  # noqa: E402\n    return ts\n\n\nCHUNK = 40          # 與正式設定 TYPO_CHUNK 相同\nCONTEXT = 3         # 與 typo_service.CONTEXT_SEGMENTS 相同\n\n\ndef system_prompt(lang: str) -> str:\n    ts = typo_service()\n    return ts.TYPO_SYSTEM_PROMPT_EN if lang == "en" else ts.TYPO_SYSTEM_PROMPT\n\n\ndef user_message(segs: list[dict], start: int, end: int) -> str:\n    """和 TypoChecker.check 組出來的 user 訊息一模一樣。segs 每項要有 segment_id、text。"""\n    ctx = segs[max(0, start - CONTEXT):start]\n    batch = segs[start:end]\n    return "\\n".join([f"[context] {s[\'text\']}" for s in ctx] + [f"[{s[\'segment_id\']}] {s[\'text\']}" for s in batch])\n\n\ndef none_word(lang: str) -> str:\n    return "NONE" if lang == "en" else "無"\n\n\ndef target_text(edits: list[dict], lang: str) -> str:\n    if not edits:\n        return none_word(lang)\n    edits = sorted(edits, key=lambda e: (e["segment_id"], e["offset"]))\n    return "\\n".join(f"{e[\'segment_id\']}|{e[\'wrong\']}|{e[\'suggestion\']}" for e in edits)\n\n\n# ---------------------------------------------------------------- 中文\n\n_OCC = None\n\n\ndef s2tw(text: str) -> str:\n    global _OCC\n    if _OCC is None:\n        from opencc import OpenCC\n        try:\n            _OCC = OpenCC("s2tw")\n        except Exception:\n            _OCC = OpenCC("s2tw.json")\n    return _OCC.convert(text)\n\n\n_ZH_MARK = re.compile(r"<[^>]*>|\\[[^\\]]*\\]")\n_ZH_SKIP = re.compile(r"[\\s\u3000，。、！？；：「」『』（）《》〈〉…—～·,.!?;:\'\\"()\\[\\]\\-~]")\n\n\ndef clean_zh_ref(text: str) -> str:\n    """AISHELL-4 參考答案：拿掉 <%> 之類的標記，簡轉繁（與專案 ASR 後處理相同的 s2tw）。"""\n    return s2tw(_ZH_MARK.sub("", text or "")).strip()\n\n\ndef zh_chars(text: str) -> list[tuple[str, int]]:\n    """去掉標點空白後的 (字, 在原字串的位置)。"""\n    return [(c, i) for i, c in enumerate(text) if not _ZH_SKIP.match(c)]\n\n\n# 同一個讀音、但從語音無法分辨或兩種寫法都對的字 → 不列為錯字\n_ZH_EQUIV = [set(s) for s in ["他她它祂牠", "的得地", "做作", "分份", "那哪", "嗎嘛麼", "啊阿呀哇", "欸誒", "喔哦噢", "吧罷",\n                               "裡裏里", "台臺", "著着", "妳你", "週周", "於于", "佔占", "畫劃", "麵面", "髮發", "鐘鍾", "嚐嘗", "甚什", "檯臺台", "坐座", "夥伙", "佈布", "祇只", "姊姐", "羣群", "汙污", "痴癡", "祕秘", "牆墻", "啟啓", "峰峯", "湧涌", "線綫", "叫呌", "吃喫", "睏困", "鬥斗", "麼么", "兒而"]]\n_ZH_FILLER = set("的了啊呀吧嗎呢喔哦欸嗯呃哎唉噢啦嘛耶呵哈么麼")\n_INIT_NEAR = [{"zh", "z"}, {"ch", "c"}, {"sh", "s"}, {"n", "l"}, {"f", "h"}, {"r", "l"}]\n_FINAL_NEAR = [{"an", "ang"}, {"en", "eng"}, {"in", "ing"}, {"ian", "iang"}, {"uan", "uang"}, {"on", "ong"}]\n\n\n@lru_cache(maxsize=100000)\ndef _py(c: str) -> tuple[str, str, str]:\n    from pypinyin import Style, pinyin\n    full = pinyin(c, style=Style.NORMAL, heteronym=False)[0][0]\n    ini = pinyin(c, style=Style.INITIALS, strict=False)[0][0]\n    fin = pinyin(c, style=Style.FINALS, strict=False)[0][0]\n    return full, ini, fin\n\n\ndef zh_sound_alike(a: str, b: str) -> bool:\n    if a == b or not ("一" <= a <= "鿿" and "一" <= b <= "鿿"):\n        return False\n    pa, ia, fa = _py(a)\n    pb, ib, fb = _py(b)\n    if pa == pb:\n        return True\n    if ia == ib and {fa, fb} in _FINAL_NEAR:\n        return True\n    if fa == fb and {ia, ib} in _INIT_NEAR:\n        return True\n    return False\n\n\ndef _zh_equiv(a: str, b: str) -> bool:\n    return any(a in s and b in s for s in _ZH_EQUIV)\n\n\ndef _unique_span(text: str, lo: int, hi: int, max_len: int = 12) -> tuple[int, int] | None:\n    """從 [lo, hi) 往兩邊各加一個字；如果在段落中不唯一就再往外擴，最多 max_len 字。"""\n    lo2, hi2 = max(0, lo - 1), min(len(text), hi + 1)\n    while True:\n        if hi2 - lo2 > max_len:\n            return None\n        if text.find(text[lo2:hi2]) == lo2 and text.find(text[lo2:hi2], lo2 + 1) < 0:\n            return lo2, hi2\n        if lo2 == 0 and hi2 == len(text):\n            return None\n        lo2, hi2 = max(0, lo2 - 1), min(len(text), hi2 + 1)\n\n\ndef zh_edits(hyp: str, ref: str, segment_id: str) -> list[dict]:\n    """找出 hyp 中「同音／近音、等長、1～3 字」的替換錯誤，輸出成正式格式的一行一行建議。"""\n    hc, rc = zh_chars(hyp), zh_chars(ref)\n    a, b = [c for c, _ in hc], [c for c, _ in rc]\n    out = []\n    sm = difflib.SequenceMatcher(None, a, b, autojunk=False)\n    for op, i1, i2, j1, j2 in sm.get_opcodes():\n        if op != "replace" or i2 - i1 != j2 - j1 or not 1 <= i2 - i1 <= 3:\n            continue\n        pairs = list(zip(a[i1:i2], b[j1:j2]))\n        if any(x in _ZH_FILLER or y in _ZH_FILLER or _zh_equiv(x, y) or not zh_sound_alike(x, y) for x, y in pairs):\n            continue\n        pos = [hc[k][1] for k in range(i1, i2)]\n        span = _unique_span(hyp, pos[0], pos[-1] + 1)\n        if span is None:\n            continue\n        lo, hi = span\n        fixed = list(hyp[lo:hi])\n        for p, (_, y) in zip(pos, pairs):\n            fixed[p - lo] = y\n        out.append({"segment_id": segment_id, "wrong": hyp[lo:hi], "suggestion": "".join(fixed),\n                    "offset": lo, "source": "asr"})\n    return out\n\n\n# ---------------------------------------------------------------- 英文\n\n_EN_FILLER = {"um", "uh", "uhm", "hmm", "mm", "mm-hmm", "mmhmm", "uh-huh", "ah", "oh", "er", "erm", "eh", "huh", "yeah",\n              "yep", "yes", "okay", "ok", "alright", "right", "so", "like", "well", "mhm", "hm", "nah", "no", "wow", "ooh"}\n_EN_VARIANT = [{"gonna", "going"}, {"wanna", "want"}, {"gotta", "got"}, {"cause", "because"}, {"ok", "okay"},\n               {"til", "until"}, {"em", "them"}, {"kinda", "kind"}, {"sorta", "sort"}, {"dunno", "know"}]\n_EN_STOP = {"a", "an", "the", "and", "or", "but", "of", "to", "in", "on", "at", "for", "is", "are", "was", "were", "be",\n            "it", "its", "it\'s", "this", "that", "i", "you", "we", "they", "he", "she", "do", "does", "did", "have",\n            "has", "had", "with", "as", "if", "then", "than", "there", "their", "they\'re", "what", "which", "who",\n            "not", "can", "will", "would", "could", "should", "just", "um", "uh"}\n_EN_TOK = re.compile(r"[^a-z0-9\']+")\n\n\ndef en_norm(tok: str) -> str:\n    return _EN_TOK.sub("", tok.lower()).strip("\'")\n\n\ndef _en_sound_alike(a: str, b: str) -> bool:\n    import jellyfish\n    if a == b:\n        return False\n    ma, mb = jellyfish.metaphone(a), jellyfish.metaphone(b)\n    if not ma or not mb:\n        return False\n    return jellyfish.jaro_winkler_similarity(ma, mb) >= 0.75 and jellyfish.levenshtein_distance(a, b) <= max(len(a), len(b)) * 0.6\n\n\ndef _match_case(src: str, word: str) -> str:\n    if src.isupper() and len(src) > 1:\n        return word.upper()\n    if src[:1].isupper():\n        return word[:1].upper() + word[1:]\n    return word\n\n\ndef en_edits(hyp: str, ref: str, segment_id: str) -> list[dict]:\n    raw = hyp.split()\n    keep = [(en_norm(t), k) for k, t in enumerate(raw) if en_norm(t)]\n    a = [w for w, _ in keep]\n    b = [en_norm(t) for t in ref.split() if en_norm(t)]\n    out = []\n    for op, i1, i2, j1, j2 in difflib.SequenceMatcher(None, a, b, autojunk=False).get_opcodes():\n        if op != "replace" or not (1 <= i2 - i1 <= 3 and 1 <= j2 - j1 <= 3):\n            continue\n        wa, wb = a[i1:i2], b[j1:j2]\n        allw = wa + wb\n        if any(w in _EN_FILLER or w.endswith("-") or any(ch.isdigit() for ch in w) for w in allw):\n            continue\n        if any({x, y} in _EN_VARIANT for x in wa for y in wb):\n            continue\n        if all(w in _EN_STOP for w in allw):\n            continue\n        if not _en_sound_alike("".join(wa), "".join(wb)):\n            continue\n        rk = [keep[k][1] for k in range(i1, i2)]\n        lo_k, hi_k = max(0, rk[0] - 1), min(len(raw), rk[-1] + 2)\n        if hi_k - lo_k > 8:\n            continue\n        before, after = raw[lo_k:rk[0]], raw[rk[-1] + 1:hi_k]\n        changed = raw[rk[0]:rk[-1] + 1]\n        lead = re.match(r"^\\W*", changed[0]).group(0)\n        trail = re.search(r"\\W*$", changed[-1]).group(0)\n        new = [_match_case(changed[min(n, len(changed) - 1)].strip(".,!?;:\\"\'"), w) for n, w in enumerate(wb)]\n        new[0] = lead + new[0]\n        new[-1] = new[-1] + trail\n        wrong = " ".join(before + changed + after)\n        sugg = " ".join(before + new + after)\n        # 原文片段在段落中要能唯一找到\n        offset = hyp.find(wrong)\n        if offset < 0 or hyp.find(wrong, offset + 1) >= 0 or wrong.lower() == sugg.lower():\n            continue\n        out.append({"segment_id": segment_id, "wrong": wrong, "suggestion": sugg, "offset": offset, "source": "asr"})\n    return out\n\n\ndef edits_for(lang: str, hyp: str, ref: str, segment_id: str) -> list[dict]:\n    return en_edits(hyp, ref, segment_id) if lang == "en" else zh_edits(hyp, ref, segment_id)\n\n\n# ---------------------------------------------------------------- 規則式同音字注入（擴充正例）\n\nclass ZhInjector:\n    """用 jieba 詞典建立「同音詞」表，把句子中的詞換成另一個同音詞（例如 效率→校率 這類不存在就跳過）。"""\n\n    def __init__(self, min_freq: int = 200):\n        import jieba\n        from pypinyin import Style, lazy_pinyin\n        groups: dict[str, list[tuple[str, int]]] = {}\n        dict_path = Path(jieba.__file__).parent / "dict.txt"\n        for line in dict_path.read_text(encoding="utf-8").splitlines():\n            parts = line.split()\n            if len(parts) < 2:\n                continue\n            w, f = parts[0], int(parts[1])\n            if not 2 <= len(w) <= 3 or f < min_freq or not all("一" <= c <= "鿿" for c in w):\n                continue\n            w = s2tw(w)\n            key = " ".join(lazy_pinyin(w, style=Style.TONE3))\n            groups.setdefault(key, []).append((w, f))\n        self.alt: dict[str, list[str]] = {}\n        for items in groups.values():\n            words = sorted({w for w, _ in items})\n            if len(words) < 2:\n                continue\n            for w in words:\n                self.alt[w] = [x for x in words if x != w and sum(c1 != c2 for c1, c2 in zip(w, x)) <= 2\n                               and not any(_zh_equiv(c1, c2) for c1, c2 in zip(w, x) if c1 != c2)]\n        import jieba as _j\n        self._cut = _j.lcut\n\n    def inject(self, text: str, segment_id: str, rng: random.Random) -> tuple[str, dict] | None:\n        words = self._cut(text)\n        cands = [k for k, w in enumerate(words) if self.alt.get(w)]\n        rng.shuffle(cands)\n        for k in cands:\n            new_w = rng.choice(self.alt[words[k]])\n            lo = sum(len(w) for w in words[:k])\n            new_text = text[:lo] + new_w + text[lo + len(words[k]):]\n            pos = [lo + i for i, (c1, c2) in enumerate(zip(words[k], new_w)) if c1 != c2]\n            span = _unique_span(new_text, pos[0], pos[-1] + 1)\n            if span is None:\n                continue\n            s, e = span\n            fixed = list(new_text[s:e])\n            for p in pos:\n                fixed[p - s] = text[p]\n            return new_text, {"segment_id": segment_id, "wrong": new_text[s:e], "suggestion": "".join(fixed),\n                              "offset": s, "source": "synthetic"}\n        return None\n\n\nclass EnInjector:\n    """用會議語料自己的詞彙，把一個實詞換成發音相近的另一個實詞（university ↔ universal 這一類）。"""\n\n    def __init__(self, vocab_counts: dict[str, int], min_count: int = 5):\n        import jellyfish\n        words = [w for w, c in vocab_counts.items() if c >= min_count and len(w) >= 4 and w.isalpha()\n                 and w not in _EN_STOP and w not in _EN_FILLER]\n        by_code: dict[str, list[str]] = {}\n        for w in words:\n            by_code.setdefault(jellyfish.metaphone(w)[:4], []).append(w)\n        self.alt: dict[str, list[str]] = {}\n        for ws in by_code.values():\n            for w in ws:\n                alts = [x for x in ws if x != w and not x.startswith(w[:-1]) and not w.startswith(x[:-1])\n                        and jellyfish.levenshtein_distance(w, x) <= 3 and _en_sound_alike(w, x)]\n                if alts:\n                    self.alt[w] = alts\n\n    def inject(self, text: str, segment_id: str, rng: random.Random) -> tuple[str, dict] | None:\n        raw = text.split()\n        cands = [k for k, t in enumerate(raw) if self.alt.get(en_norm(t))]\n        rng.shuffle(cands)\n        for k in cands:\n            old = raw[k]\n            core = en_norm(old)\n            new_core = _match_case(old.strip(".,!?;:\\"\'"), rng.choice(self.alt[core]))\n            m = re.match(r"^(\\W*)(.*?)(\\W*)$", old)\n            new_tok = m.group(1) + new_core + m.group(3)\n            new_raw = raw[:k] + [new_tok] + raw[k + 1:]\n            lo_k, hi_k = max(0, k - 1), min(len(raw), k + 2)\n            wrong = " ".join(new_raw[lo_k:hi_k])\n            sugg = " ".join(raw[lo_k:hi_k])\n            new_text = " ".join(new_raw)\n            off = new_text.find(wrong)\n            if off < 0 or new_text.find(wrong, off + 1) >= 0:\n                continue\n            return new_text, {"segment_id": segment_id, "wrong": wrong, "suggestion": sugg, "offset": off,\n                              "source": "synthetic"}\n        return None\n\n\n# ---------------------------------------------------------------- 評分\n\ndef _apply(text: str, offset: int, wrong: str, sugg: str) -> str:\n    return text[:offset] + sugg + text[offset + len(wrong):]\n\n\ndef _norm_for_match(lang: str, text: str) -> str:\n    if lang == "en":\n        return " ".join(en_norm(t) for t in text.split() if en_norm(t))\n    return "".join(c for c, _ in zh_chars(text))\n\n\ndef score_chunk(lang: str, seg_text: dict[str, str], gold: list[dict], pred: list[dict]) -> dict:\n    """pred 用正式解析器（parse_reply / parse_reply_en）的輸出。\n    一筆建議算「對」：套用後的段落，和「只套用某一個標準答案」後的段落完全相同（忽略標點／大小寫）。"""\n    gold_res = {}\n    for gi, g in enumerate(gold):\n        t = seg_text[g["segment_id"]]\n        gold_res[gi] = (g["segment_id"], _norm_for_match(lang, _apply(t, g["offset"], g["wrong"], g["suggestion"])))\n    matched, tp = set(), 0\n    for p in pred:\n        t = seg_text.get(p["segment_id"])\n        if t is None:\n            continue\n        res = _norm_for_match(lang, _apply(t, p["offset"], p["wrong"], p["suggestion"]))\n        hit = next((gi for gi, (sid, r) in gold_res.items() if gi not in matched and sid == p["segment_id"] and r == res), None)\n        if hit is not None:\n            matched.add(hit)\n            tp += 1\n    return {"tp": tp, "pred": len(pred), "gold": len(gold)}\n\n\ndef summarize(rows: list[dict]) -> dict:\n    tp = sum(r["tp"] for r in rows)\n    pred = sum(r["pred"] for r in rows)\n    gold = sum(r["gold"] for r in rows)\n    p = tp / pred if pred else 0.0\n    r = tp / gold if gold else 0.0\n    f05 = (1.25 * p * r / (0.25 * p + r)) if (p + r) else 0.0\n    clean = [x for x in rows if x["gold"] == 0]\n    false_alarm = sum(1 for x in clean if x["pred"] > 0) / len(clean) if clean else 0.0\n    return {"chunks": len(rows), "gold": gold, "pred": pred, "tp": tp, "precision": round(p, 4),\n            "recall": round(r, 4), "f0.5": round(f05, 4), "clean_chunks": len(clean),\n            "false_alarm_rate_on_clean_chunks": round(false_alarm, 4)}\n\n\ndef error_rate(lang: str, hyp: str, ref: str) -> tuple[int, int]:\n    """回傳 (編輯距離, 參考長度)：中文用字元（CER），英文用詞（WER）。"""\n    if lang == "en":\n        a = [en_norm(t) for t in hyp.split() if en_norm(t)]\n        b = [en_norm(t) for t in ref.split() if en_norm(t)]\n    else:\n        a = [c for c, _ in zh_chars(hyp)]\n        b = [c for c, _ in zh_chars(ref)]\n    prev = list(range(len(b) + 1))\n    for i, x in enumerate(a, 1):\n        cur = [i] + [0] * len(b)\n        for j, y in enumerate(b, 1):\n            cur[j] = min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (x != y))\n        prev = cur\n    return prev[-1], len(b)\n', 'lm_utils.py': '"""訓練與評測共用：組出和推論時完全相同的 prompt 文字（關閉 thinking）。"""\nfrom __future__ import annotations\n\n\ndef chat_prompt(tok, system: str, user: str) -> str:\n    msgs = [{"role": "system", "content": system}, {"role": "user", "content": user}]\n    try:\n        return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)\n    except TypeError:\n        return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)\n\n\ndef end_text(tok) -> str:\n    """回答結尾：Qwen 系列是 <|im_end|>。"""\n    eos = tok.eos_token or "<|im_end|>"\n    return "<|im_end|>" if "<|im_end|>" in tok.get_vocab() else eos\n\n\ndef load_tokenizer(path: str):\n    from transformers import AutoTokenizer\n    tok = AutoTokenizer.from_pretrained(path, trust_remote_code=True)\n    if tok.pad_token is None:\n        tok.pad_token = tok.eos_token\n    return tok\n\n\ndef load_model(path: str, adapter: str | None = None, dtype: str = "bfloat16", load_4bit: bool = False):\n    import torch\n    from transformers import AutoConfig\n    cfg = AutoConfig.from_pretrained(path, trust_remote_code=True)\n    kw = {"dtype": getattr(torch, dtype), "device_map": "auto", "trust_remote_code": True, "low_cpu_mem_usage": True}\n    if load_4bit:\n        from transformers import BitsAndBytesConfig\n        kw["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",\n                                                       bnb_4bit_compute_dtype=getattr(torch, dtype),\n                                                       bnb_4bit_use_double_quant=True)\n    if getattr(cfg, "vision_config", None) is not None:\n        from transformers import AutoModelForImageTextToText as Auto\n    else:\n        from transformers import AutoModelForCausalLM as Auto\n    model = Auto.from_pretrained(path, **kw)\n    if adapter:\n        from peft import PeftModel\n        model = PeftModel.from_pretrained(model, adapter)\n    model.eval()\n    return model\n', 'build_asr_data.py': '"""步驟 1（Colab GPU）：下載公開會議語料 → 用「專案正式的 ASR 模型與後處理」辨識 → 存成 hyp/ref 成對的會議檔。\n\n國語：AISHELL-4（真實會議錄音、人工逐字稿，Apache-2.0）\n英文：AMI Meeting Corpus 的 sdm（單一遠距麥克風，最像實際會議錄音，CC BY 4.0）\n\n輸出：{OUT}/meetings_{lang}_{split}.jsonl，每行一場會議\n  {"meeting_id", "lang", "split", "segments": [{"segment_id": "S0001", "hyp": ..., "ref": ...}]}\n已經做過的會議會跳過（Colab 斷線後重跑會接著做）。\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport io\nimport os\nimport json\nimport re\nimport sys\nimport time\nfrom pathlib import Path\n\nimport numpy as np\n\nsys.path.insert(0, str(Path(__file__).resolve().parent))\nimport ft_common as F  # noqa: E402\n\nSR = 16000\nMERGE_GAP_S = 1.0     # 與專案 ASR_MERGE_GAP_S 相同\nMERGE_MAX_S = 20.0    # 與專案 ASR_MERGE_MAX_S 相同\n\n# 會議切分：split → [(資料夾, 第幾場起, 第幾場止)]；dev 與 train 取 train_M 不重疊的部分\nAISHELL_SPLITS = {"train": [("train_L", 0, 30), ("train_S", 0, 50), ("train_M", 4, 20)],\n                  "dev": [("train_M", 0, 4)], "test": [("test", 0, 12)]}\nAMI_SPLITS = {"train": ("train", 10), "dev": ("validation", 1), "test": ("test", 4)}   # (HF split, 取幾個 parquet 分片)\n\n\ndef log(*a):\n    print(time.strftime("%H:%M:%S"), *a, flush=True)\n\n\ndef done_ids(path: Path) -> set[str]:\n    if not path.exists():\n        return set()\n    return {json.loads(line)["meeting_id"] for line in path.open(encoding="utf-8") if line.strip()}\n\n\ndef merge_intervals(items: list[dict]) -> list[list[dict]]:\n    """items: [{start, end, ref, ...}] 已依時間排序、且不重疊。依專案規則合併成段落。"""\n    groups, cur = [], []\n    for it in items:\n        if cur and (it["start"] - cur[-1]["end"] <= MERGE_GAP_S) and (it["end"] - cur[0]["start"] <= MERGE_MAX_S):\n            cur.append(it)\n        else:\n            if cur:\n                groups.append(cur)\n            cur = [it]\n    if cur:\n        groups.append(cur)\n    return groups\n\n\ndef drop_overlaps(items: list[dict]) -> list[dict]:\n    """多人同時講話的片段，參考答案的先後順序不明確 → 去掉。"""\n    items = sorted(items, key=lambda x: x["start"])\n    keep = []\n    for i, it in enumerate(items):\n        prev_end = max((x["end"] for x in items[max(0, i - 8):i]), default=-1)\n        nxt = items[i + 1]["start"] if i + 1 < len(items) else 1e9\n        if it["start"] >= prev_end and it["end"] <= nxt:\n            keep.append(it)\n    return keep\n\n\nclass ProjectASR:\n    """直接用專案的 build_engine / is_hallucination / to_traditional，辨識結果和正式系統一致。"""\n\n    def __init__(self, lang: str, batch_size: int):\n        b = str(F.project_backend())\n        if b not in sys.path:\n            sys.path.insert(0, b)\n        from app.services import asr_models\n        from app.services.asr_service import is_hallucination, to_traditional\n        self.lang = lang\n        self.engine = asr_models.build_engine(lang)\n        inner = getattr(self.engine, "_engine", None)\n        if inner is not None:\n            inner.batch_size = batch_size\n        self.is_hallucination, self.to_traditional = is_hallucination, to_traditional\n        log("載入 ASR：", asr_models.LANGUAGES[lang]["model_id"])\n        self.engine.load()\n\n    def run(self, clips: list[np.ndarray], step: int = 64) -> list[str | None]:\n        texts = []\n        for i in range(0, len(clips), step):     # 分批送，避免一場會議幾百段一次塞爆顯示卡\n            texts += self.engine.transcribe_clips(clips[i:i + step], SR)\n        out = []\n        for clip, t in zip(clips, texts):\n            if self.is_hallucination(t, len(clip) / SR):\n                out.append(None)\n                continue\n            if self.lang != "en":\n                t = self.to_traditional(t)\n            out.append(t.strip() or None)\n        return out\n\n    def unload(self):\n        self.engine.unload()\n\n\ndef build_meeting(meeting_id: str, lang: str, split: str, groups: list[list[dict]], clips, asr: ProjectASR) -> dict:\n    hyps = asr.run(clips)\n    segs = []\n    for g, hyp in zip(groups, hyps):\n        ref = " ".join(x["ref"] for x in g) if lang == "en" else "".join(x["ref"] for x in g)\n        if not hyp or not ref.strip():\n            continue\n        segs.append({"segment_id": f"S{len(segs) + 1:04d}", "hyp": hyp, "ref": ref,\n                     "start": round(g[0]["start"], 2), "end": round(g[-1]["end"], 2)})\n    return {"meeting_id": meeting_id, "lang": lang, "split": split, "segments": segs}\n\n\n# ------------------------------------------------------------------ AISHELL-4\n\ndef parse_textgrid(text: str) -> list[dict]:\n    items = []\n    for tier in re.split(r"item \\[\\d+\\]:", text)[1:]:\n        for m in re.finditer(r"xmin = ([\\d.]+)\\s*\\n\\s*xmax = ([\\d.]+)\\s*\\n\\s*text = \\"(.*?)\\"", tier, flags=re.S):\n            ref = F.clean_zh_ref(m.group(3))\n            if len(F.zh_chars(ref)) >= 2:\n                items.append({"start": float(m.group(1)), "end": float(m.group(2)), "ref": ref})\n    return items\n\n\ndef run_aishell(out: Path, splits: list[str], batch_size: int, limit_scale: float):\n    import soundfile as sf\n    from huggingface_hub import HfApi, hf_hub_download\n    repo = "AISHELL/AISHELL-4"\n    files = [f for f in HfApi().list_repo_files(repo, repo_type="dataset")]\n    asr = None\n    for split in splits:\n        path = out / f"meetings_zh_{split}.jsonl"\n        have = done_ids(path)\n        for folder, lo, hi in AISHELL_SPLITS[split]:\n            hi = lo + max(1, int((hi - lo) * limit_scale))\n            tgs = sorted(f for f in files if f.startswith(f"{folder}/TextGrid/") and f.endswith(".TextGrid"))[lo:hi]\n            for tg in tgs:\n                mid = Path(tg).stem\n                if mid in have:\n                    continue\n                flac = next((f for f in files if f.startswith(f"{folder}/") and Path(f).stem == mid and f.endswith(".flac")), None)\n                if flac is None:\n                    log("找不到音檔，略過", mid)\n                    continue\n                dl = Path(os.environ.get("FT_DOWNLOAD", "/tmp/ft_download"))\n                items = drop_overlaps(parse_textgrid(Path(hf_hub_download(repo, tg, repo_type="dataset", local_dir=dl)).read_text(encoding="utf-8")))\n                flac_path = Path(hf_hub_download(repo, flac, repo_type="dataset", local_dir=dl))\n                audio, sr = sf.read(str(flac_path), dtype="float32")\n                if audio.ndim > 1:\n                    # AISHELL-4 是 8 支麥克風的陣列；直接平均會讓聲音相位互相抵消、辨識變很差。\n                    # 取其中一支麥克風 = 一台放在會議桌上的手機或筆電錄音，比較接近實際使用情境。\n                    audio = audio[:, 0]\n                if sr != SR:\n                    import librosa\n                    audio = librosa.resample(audio, orig_sr=sr, target_sr=SR)\n                groups = merge_intervals(items)\n                clips = [audio[int(g[0]["start"] * SR): int(g[-1]["end"] * SR)] for g in groups]\n                if asr is None:\n                    asr = ProjectASR("zh", batch_size)\n                t0 = time.time()\n                rec = build_meeting(mid, "zh", split, groups, clips, asr)\n                with path.open("a", encoding="utf-8") as f:\n                    f.write(json.dumps(rec, ensure_ascii=False) + "\\n")\n                log(f"[zh/{split}] {mid}：{len(rec[\'segments\'])} 段，{len(audio) / SR / 60:.0f} 分鐘音檔，辨識 {time.time() - t0:.0f}s")\n                flac_path.unlink(missing_ok=True)   # 省磁碟\n    if asr:\n        asr.unload()\n\n\n# ------------------------------------------------------------------ AMI\n\ndef run_ami(out: Path, splits: list[str], batch_size: int, limit_scale: float):\n    import pyarrow.parquet as pq\n    import soundfile as sf\n    from huggingface_hub import HfApi, hf_hub_download\n    repo = "edinburghcstr/ami"\n    files = HfApi().list_repo_files(repo, repo_type="dataset")\n    asr = None\n    for split in splits:\n        hf_split, n = AMI_SPLITS[split]\n        n = max(1, int(n * limit_scale))\n        path = out / f"meetings_en_{split}.jsonl"\n        have = done_ids(path)\n        shards = sorted(f for f in files if f.startswith(f"sdm/{hf_split}-"))[:n]\n        by_meeting: dict[str, list[dict]] = {}\n        for sh in shards:\n            t = pq.read_table(hf_hub_download(repo, sh, repo_type="dataset"))\n            for row in t.to_pylist():\n                if row["meeting_id"] in have:\n                    continue\n                by_meeting.setdefault(row["meeting_id"], []).append(\n                    {"start": row["begin_time"], "end": row["end_time"], "ref": row["text"], "audio": row["audio"]["bytes"]})\n        for mid, items in sorted(by_meeting.items()):\n            items = drop_overlaps(items)\n            groups = merge_intervals(items)\n            clips = []\n            for g in groups:\n                parts = []\n                for k, x in enumerate(g):\n                    if k:\n                        parts.append(np.zeros(int(min(0.3, max(0.0, x["start"] - g[k - 1]["end"])) * SR), dtype=np.float32))\n                    a, sr = sf.read(io.BytesIO(x["audio"]), dtype="float32")\n                    if a.ndim > 1:\n                        a = a.mean(axis=1)\n                    if sr != SR:\n                        import librosa\n                        a = librosa.resample(a, orig_sr=sr, target_sr=SR)\n                    parts.append(a)\n                clips.append(np.concatenate(parts))\n            if asr is None:\n                asr = ProjectASR("en", batch_size)\n            t0 = time.time()\n            rec = build_meeting(mid, "en", split, groups, clips, asr)\n            with path.open("a", encoding="utf-8") as f:\n                f.write(json.dumps(rec, ensure_ascii=False) + "\\n")\n            log(f"[en/{split}] {mid}：{len(rec[\'segments\'])} 段，辨識 {time.time() - t0:.0f}s")\n    if asr:\n        asr.unload()\n\n\nif __name__ == "__main__":\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--out", required=True)\n    ap.add_argument("--langs", default="zh,en")\n    ap.add_argument("--splits", default="train,dev,test")\n    ap.add_argument("--batch-size", type=int, default=16)\n    ap.add_argument("--scale", type=float, default=1.0, help="會議數量倍率（測試用 0.1）")\n    a = ap.parse_args()\n    out = Path(a.out)\n    out.mkdir(parents=True, exist_ok=True)\n    splits = a.splits.split(",")\n    if "zh" in a.langs:\n        run_aishell(out, splits, a.batch_size, a.scale)\n    if "en" in a.langs:\n        run_ami(out, splits, a.batch_size, a.scale)\n    log("完成")\n', 'build_sft.py': '"""步驟 2（CPU 即可）：把 hyp/ref 會議檔轉成訓練／評測樣本。\n\n每個樣本 = 正式系統送給 Qwen 的一批（40 段 + 前面 3 段 [context]），格式與 typo_service 完全相同；\n標準答案 = 人工逐字稿與 ASR 結果之間「同音／近音替換」的差異（＋train 才有的規則式同音字注入）。\n所有標準答案都會先用正式的 parse_reply / parse_reply_en 跑過一次，解析不出來的不收。\n\n輸出：{DATA}/sft_train.jsonl、sft_dev.jsonl、sft_test.jsonl、stats.json\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport collections\nimport json\nimport random\nimport sys\nfrom pathlib import Path\n\nsys.path.insert(0, str(Path(__file__).resolve().parent))\nimport ft_common as F  # noqa: E402\n\nINJECT_RATE = {"zh": 0.10, "en": 0.06}   # train 擴充版本中，沒有真實錯字的段落被注入同音字的比例\n\n\ndef load_meetings(data: Path, lang: str, split: str) -> list[dict]:\n    p = data / f"meetings_{lang}_{split}.jsonl"\n    if not p.exists():\n        return []\n    return [json.loads(x) for x in p.open(encoding="utf-8") if x.strip()]\n\n\ndef verified(lang: str, seg_text: dict[str, str], edits: list[dict]) -> list[dict]:\n    """只保留正式解析器解析得出、而且位置一致的標準答案。"""\n    ts = F.typo_service()\n    ok = []\n    for e in edits:\n        line = F.target_text([e], lang)\n        parsed = (ts.parse_reply_en if lang == "en" else ts.parse_reply)(line, {e["segment_id"]: seg_text[e["segment_id"]]})\n        if parsed and parsed[0]["offset"] == e["offset"]:\n            ok.append(e)\n    return ok\n\n\ndef meeting_examples(m: dict, rng: random.Random, augment: bool, offsets: list[int], injector) -> list[dict]:\n    lang = m["lang"]\n    segs = [{"segment_id": s["segment_id"], "text": s["hyp"], "ref": s["ref"]} for s in m["segments"]]\n    seg_text = {s["segment_id"]: s["text"] for s in segs}\n    gold = {s["segment_id"]: verified(lang, seg_text, F.edits_for(lang, s["text"], s["ref"], s["segment_id"])) for s in segs}\n    variants = [("real", segs, gold)]\n    if augment and injector is not None:\n        segs2, gold2 = [], {}\n        for s in segs:\n            sid = s["segment_id"]\n            s2 = dict(s)\n            g = list(gold[sid])\n            if not g and rng.random() < INJECT_RATE[lang]:\n                r = injector.inject(s["text"], sid, rng)\n                if r:\n                    new_text, e = r\n                    if verified(lang, {sid: new_text}, [e]):\n                        s2["text"], g = new_text, [e]\n            segs2.append(s2)\n            gold2[sid] = g\n        variants.append(("aug", segs2, gold2))\n    out = []\n    for kind, ss, gg in variants:\n        for off in offsets:\n            for start in range(off, len(ss), F.CHUNK):\n                end = min(len(ss), start + F.CHUNK)\n                if end - start < 5:\n                    continue\n                batch = ss[start:end]\n                edits = [e for s in batch for e in gg[s["segment_id"]]]\n                out.append({\n                    "id": f"{m[\'meeting_id\']}:{kind}:{start}", "lang": lang, "split": m["split"], "kind": kind,\n                    "system": F.system_prompt(lang), "user": F.user_message(ss, start, end),\n                    "target": F.target_text(edits, lang),\n                    "seg_text": {s["segment_id"]: s["text"] for s in batch},\n                    "refs": {s["segment_id"]: s["ref"] for s in batch},\n                    "gold": edits,\n                })\n    return out\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--data", required=True)\n    ap.add_argument("--langs", default="zh,en")\n    ap.add_argument("--seed", type=int, default=42)\n    a = ap.parse_args()\n    data = Path(a.data)\n    rng = random.Random(a.seed)\n    stats = {}\n    for split in ["train", "dev", "test"]:\n        rows = []\n        for lang in a.langs.split(","):\n            meetings = load_meetings(data, lang, split)\n            if not meetings:\n                print(f"（沒有 {lang}/{split} 的會議檔，略過）")\n                continue\n            injector = None\n            if split == "train":\n                if lang == "zh":\n                    injector = F.ZhInjector(min_freq=50)\n                else:\n                    vocab = collections.Counter(F.en_norm(t) for m in meetings for s in m["segments"] for t in s["ref"].split())\n                    injector = F.EnInjector(vocab)\n            for m in meetings:\n                rows += meeting_examples(m, rng, augment=(split == "train"),\n                                         offsets=[0, F.CHUNK // 2] if split == "train" else [0], injector=injector)\n        rng.shuffle(rows)\n        with (data / f"sft_{split}.jsonl").open("w", encoding="utf-8") as f:\n            for r in rows:\n                f.write(json.dumps(r, ensure_ascii=False) + "\\n")\n        st = {}\n        for lang in a.langs.split(","):\n            rr = [r for r in rows if r["lang"] == lang]\n            if not rr:\n                continue\n            src = collections.Counter(e["source"] for r in rr for e in r["gold"])\n            st[lang] = {"examples": len(rr), "clean_examples": sum(1 for r in rr if not r["gold"]),\n                        "edits": sum(len(r["gold"]) for r in rr), "by_source": dict(src),\n                        "kinds": dict(collections.Counter(r["kind"] for r in rr))}\n        stats[split] = st\n        print(split, json.dumps(st, ensure_ascii=False))\n    (data / "stats.json").write_text(json.dumps(stats, ensure_ascii=False, indent=2), encoding="utf-8")\n\n\nif __name__ == "__main__":\n    main()\n', 'eval_typo.py': '"""評分：把測試樣本送給模型，用正式解析器解析，跟標準答案比較。\n\n後端：\n  --backend ollama  走和正式系統一模一樣的 /api/chat 呼叫（最終成績以這個為準）\n  --backend hf      transformers 直接推論，可掛 LoRA adapter（訓練中快速比較用）\n  --backend gold    直接拿標準答案當模型輸出（檢查評分程式本身，應該 100%）\n\n輸出：{OUT}/{name}.json（各項分數）與 {name}.predictions.jsonl（每批原始回覆，方便人工檢查）\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport re\nimport sys\nimport time\nfrom pathlib import Path\n\nsys.path.insert(0, str(Path(__file__).resolve().parent))\nimport ft_common as F  # noqa: E402\n\n\ndef apply_all(text: str, items: list[dict]) -> str:\n    for it in sorted(items, key=lambda x: -x["offset"]):\n        if text[it["offset"]:it["offset"] + len(it["wrong"])] == it["wrong"]:\n            text = text[:it["offset"]] + it["suggestion"] + text[it["offset"] + len(it["wrong"]):]\n    return text\n\n\nclass OllamaBackend:\n    def __init__(self, model: str, host: str):\n        F.typo_service()          # 把專案 backend 加進 sys.path\n        from app import config  # noqa: F401  ← 讀 .env，溫度／seed／num_ctx 和正式設定相同\n        self.model, self.host = model, host\n\n    def generate(self, batch: list[dict]) -> list[str]:\n        import httpx\n        from app import config\n        out = []\n        for ex in batch:\n            payload = {\n                "model": self.model,\n                "messages": [{"role": "system", "content": ex["system"]}, {"role": "user", "content": ex["user"]}],\n                "stream": False, "think": config.TYPO_THINK,\n                "options": {"temperature": config.LLM_TEMPERATURE, "seed": config.LLM_SEED, "num_ctx": config.LLM_NUM_CTX},\n            }\n            r = httpx.post(f"{self.host}/api/chat", json=payload, timeout=config.LLM_TIMEOUT_SECONDS)\n            r.raise_for_status()\n            content = (r.json().get("message") or {}).get("content") or ""\n            out.append(re.sub(r"<think>.*?</think>", "", content, flags=re.S))\n        return out\n\n\nclass HFBackend:\n    def __init__(self, base: str, adapter: str | None, max_new_tokens: int = 512):\n        import lm_utils as L\n        self.L = L\n        self.tok = L.load_tokenizer(base)\n        self.tok.padding_side = "left"\n        self.model = L.load_model(base, adapter)\n        self.max_new_tokens = max_new_tokens\n\n    def generate(self, batch: list[dict]) -> list[str]:\n        import torch\n        texts = [self.L.chat_prompt(self.tok, ex["system"], ex["user"]) for ex in batch]\n        enc = self.tok(texts, return_tensors="pt", padding=True, add_special_tokens=False).to(self.model.device)\n        with torch.no_grad():\n            gen = self.model.generate(**enc, max_new_tokens=self.max_new_tokens, do_sample=False,\n                                      pad_token_id=self.tok.pad_token_id)\n        outs = self.tok.batch_decode(gen[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)\n        return [re.sub(r"<think>.*?</think>", "", o, flags=re.S).strip() for o in outs]\n\n\nclass GoldBackend:\n    def generate(self, batch):\n        return [ex["target"] for ex in batch]\n\n\ndef evaluate(examples: list[dict], backend, batch_size: int, pred_path: Path) -> dict:\n    ts = F.typo_service()\n    done = {}\n    if pred_path.exists():   # 斷線後接著跑\n        for line in pred_path.open(encoding="utf-8"):\n            r = json.loads(line)\n            done[r["id"]] = r["reply"]\n    todo = [ex for ex in examples if ex["id"] not in done]\n    t0 = time.time()\n    with pred_path.open("a", encoding="utf-8") as f:\n        for i in range(0, len(todo), batch_size):\n            batch = todo[i:i + batch_size]\n            for ex, rep in zip(batch, backend.generate(batch)):\n                done[ex["id"]] = rep\n                f.write(json.dumps({"id": ex["id"], "reply": rep}, ensure_ascii=False) + "\\n")\n            n = i + len(batch)\n            print(f"  {n}/{len(todo)}  {(time.time() - t0) / n:.1f}s/批", flush=True)\n\n    rows, groups = [], {}\n    err = {}\n    for ex in examples:\n        lang = ex["lang"]\n        reply = done[ex["id"]]\n        parser = ts.parse_reply_en if lang == "en" else ts.parse_reply\n        pred = ts.merge(parser(reply, ex["seg_text"]), {sid: k for k, sid in enumerate(ex["seg_text"])})\n        sc = F.score_chunk(lang, ex["seg_text"], ex["gold"], pred)\n        rows.append(sc)\n        for key in [lang, f"{lang}/{ex[\'kind\']}"]:\n            groups.setdefault(key, []).append(sc)\n        # 整份逐字稿的錯誤率：原始 ASR、套用模型建議、套用全部標準答案（上限）\n        e = err.setdefault(lang, {"before": [0, 0], "after_model": [0, 0], "after_gold": [0, 0]})\n        by_seg_pred, by_seg_gold = {}, {}\n        for p in pred:\n            by_seg_pred.setdefault(p["segment_id"], []).append(p)\n        for g in ex["gold"]:\n            by_seg_gold.setdefault(g["segment_id"], []).append(g)\n        for sid, text in ex["seg_text"].items():\n            ref = ex["refs"][sid]\n            for k, t in [("before", text), ("after_model", apply_all(text, by_seg_pred.get(sid, []))),\n                         ("after_gold", apply_all(text, by_seg_gold.get(sid, [])))]:\n                d, n = F.error_rate(lang, t, ref)\n                e[k][0] += d\n                e[k][1] += n\n    res = {k: F.summarize(v) for k, v in sorted(groups.items())}\n    res["all"] = F.summarize(rows)\n    for lang, e in err.items():\n        res[lang]["error_rate"] = {k: round(d / n, 4) if n else None for k, (d, n) in e.items()}\n        res[lang]["error_rate_metric"] = "WER" if lang == "en" else "CER"\n    return res\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--data", required=True, help="sft_*.jsonl 所在資料夾")\n    ap.add_argument("--split", default="test")\n    ap.add_argument("--kinds", default="real", help="real / aug / real,aug")\n    ap.add_argument("--backend", choices=["ollama", "hf", "gold"], required=True)\n    ap.add_argument("--model", default="qwen3.8:27b", help="ollama 模型名稱")\n    ap.add_argument("--host", default="http://127.0.0.1:11434")\n    ap.add_argument("--base", help="hf：基底模型路徑")\n    ap.add_argument("--adapter", help="hf：LoRA adapter 路徑")\n    ap.add_argument("--batch-size", type=int, default=8)\n    ap.add_argument("--limit", type=int, default=0)\n    ap.add_argument("--name", required=True)\n    ap.add_argument("--out", required=True)\n    a = ap.parse_args()\n\n    examples = [json.loads(x) for x in Path(a.data, f"sft_{a.split}.jsonl").open(encoding="utf-8")]\n    kinds = set(a.kinds.split(","))\n    examples = sorted([e for e in examples if e["kind"] in kinds], key=lambda e: e["id"])\n    if a.limit:\n        examples = examples[:a.limit]\n    out = Path(a.out)\n    out.mkdir(parents=True, exist_ok=True)\n    if a.backend == "ollama":\n        F.typo_service()\n        backend, bs = OllamaBackend(a.model, a.host), 1\n    elif a.backend == "hf":\n        backend, bs = HFBackend(a.base, a.adapter), a.batch_size\n    else:\n        backend, bs = GoldBackend(), 64\n    print(f"評測 {a.name}：{len(examples)} 批")\n    res = evaluate(examples, backend, bs, out / f"{a.name}.predictions.jsonl")\n    res["_meta"] = {"name": a.name, "backend": a.backend, "model": a.model if a.backend == "ollama" else a.base,\n                    "adapter": a.adapter, "split": a.split, "kinds": sorted(kinds)}\n    (out / f"{a.name}.json").write_text(json.dumps(res, ensure_ascii=False, indent=2), encoding="utf-8")\n    print(json.dumps({k: v for k, v in res.items() if k != "_meta"}, ensure_ascii=False, indent=1))\n\n\nif __name__ == "__main__":\n    main()\n', 'train_lora.py': '"""步驟 3（Colab GPU）：LoRA 微調 Qwen3.8-27B 的錯字校正能力。\n\n- 只訓練語言模型的注意力與 MLP 投影層（LoRA），原始權重不動；視覺部分完全不碰。\n- loss 只算「模型的回答」那段，prompt（系統指令＋逐字稿）不算。\n- checkpoint 存到 --out（請放在 Google Drive），斷線後重跑會自動接續。\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport math\nimport random\nimport sys\nfrom pathlib import Path\n\nsys.path.insert(0, str(Path(__file__).resolve().parent))\nimport lm_utils as L  # noqa: E402\n\n# 全注意力層：q/k/v/o；線性注意力層（Gated DeltaNet）：in_proj_qkv / in_proj_z / out_proj；每層的 MLP。\n_MODS = (r"(self_attn\\.(q_proj|k_proj|v_proj|o_proj)|linear_attn\\.(in_proj_qkv|in_proj_z|out_proj)"\n         r"|mlp\\.(gate_proj|up_proj|down_proj))$")\nTARGET = r"^(base_model\\.model\\.)?model\\.(language_model\\.)?layers\\.\\d+\\." + _MODS\n\n\ndef encode(tok, ex: dict, max_len: int):\n    prompt = L.chat_prompt(tok, ex["system"], ex["user"])\n    answer = ex["target"] + L.end_text(tok) + "\\n"\n    p_ids = tok(prompt, add_special_tokens=False)["input_ids"]\n    a_ids = tok(answer, add_special_tokens=False)["input_ids"]\n    if len(p_ids) + len(a_ids) > max_len:\n        return None\n    return {"input_ids": p_ids + a_ids, "labels": [-100] * len(p_ids) + a_ids}\n\n\nclass Collator:\n    def __init__(self, pad_id: int):\n        self.pad_id = pad_id\n\n    def __call__(self, feats):\n        import torch\n        n = max(len(f["input_ids"]) for f in feats)\n        ids = torch.full((len(feats), n), self.pad_id, dtype=torch.long)\n        lab = torch.full((len(feats), n), -100, dtype=torch.long)\n        att = torch.zeros((len(feats), n), dtype=torch.long)\n        for i, f in enumerate(feats):\n            k = len(f["input_ids"])\n            ids[i, :k] = torch.tensor(f["input_ids"])\n            lab[i, :k] = torch.tensor(f["labels"])\n            att[i, :k] = 1\n        return {"input_ids": ids, "labels": lab, "attention_mask": att}\n\n\ndef load(path: Path, kinds: set[str] | None = None) -> list[dict]:\n    rows = [json.loads(x) for x in path.open(encoding="utf-8") if x.strip()]\n    return [r for r in rows if kinds is None or r["kind"] in kinds]\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--base", required=True)\n    ap.add_argument("--data", required=True)\n    ap.add_argument("--out", required=True)\n    ap.add_argument("--epochs", type=float, default=2.0)\n    ap.add_argument("--lr", type=float, default=1e-4)\n    ap.add_argument("--rank", type=int, default=16)\n    ap.add_argument("--alpha", type=int, default=32)\n    ap.add_argument("--max-len", type=int, default=6144)\n    ap.add_argument("--grad-accum", type=int, default=16)\n    ap.add_argument("--save-steps", type=int, default=20)\n    ap.add_argument("--dev-limit", type=int, default=48)\n    ap.add_argument("--load-4bit", action="store_true", help="顯示卡記憶體不夠時改用 QLoRA")\n    ap.add_argument("--max-steps", type=int, default=-1, help="測試用")\n    ap.add_argument("--max-train", type=int, default=1600, help="訓練樣本上限（控制 Colab 時數）；0 = 全部")\n    a = ap.parse_args()\n\n    import torch\n    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training\n    from transformers import Trainer, TrainingArguments, trainer_utils\n\n    out = Path(a.out)\n    out.mkdir(parents=True, exist_ok=True)\n    tok = L.load_tokenizer(a.base)\n\n    train_rows = load(Path(a.data) / "sft_train.jsonl")\n    dev_rows = load(Path(a.data) / "sft_dev.jsonl")\n    random.Random(0).shuffle(dev_rows)\n    random.Random(1).shuffle(train_rows)\n    if a.max_train:\n        train_rows = train_rows[:a.max_train]\n    train = [e for e in (encode(tok, r, a.max_len) for r in train_rows) if e]\n    dev = [e for e in (encode(tok, r, a.max_len) for r in dev_rows) if e][:a.dev_limit]\n    lens = sorted(len(e["input_ids"]) for e in train)\n    print(f"訓練 {len(train)} 筆（超過 {a.max_len} token 丟掉 {len(train_rows) - len(train)} 筆）；"\n          f"dev {len(dev)} 筆；長度中位數 {lens[len(lens) // 2]}、最大 {lens[-1]}", flush=True)\n\n    model = L.load_model(a.base, load_4bit=a.load_4bit)\n    model.config.use_cache = False\n    if a.load_4bit:\n        model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)\n    else:\n        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})\n        model.enable_input_require_grads()\n    lcfg = LoraConfig(r=a.rank, lora_alpha=a.alpha, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",\n                      target_modules=TARGET)\n    model = get_peft_model(model, lcfg)\n    model.print_trainable_parameters()\n    names = [n for n, _ in model.named_modules() if n.endswith("lora_A")]\n    print("LoRA 掛在", len(names), "個模組，例如", names[:3], flush=True)\n    if not names:\n        raise RuntimeError("LoRA 沒有掛上任何模組，請檢查 TARGET 正規表示式")\n\n    steps_per_epoch = math.ceil(len(train) / a.grad_accum)\n    total_steps = a.max_steps if a.max_steps > 0 else math.ceil(steps_per_epoch * a.epochs)\n    args = TrainingArguments(\n        output_dir=str(out / "checkpoints"),\n        per_device_train_batch_size=1, per_device_eval_batch_size=1,\n        gradient_accumulation_steps=a.grad_accum,\n        num_train_epochs=a.epochs, max_steps=a.max_steps,\n        learning_rate=a.lr, lr_scheduler_type="cosine", warmup_steps=max(1, int(total_steps * 0.03)), weight_decay=0.0,\n        bf16=torch.cuda.is_available(), logging_steps=2,\n        eval_strategy="steps" if dev else "no", eval_steps=a.save_steps, save_strategy="steps", save_steps=a.save_steps,\n        save_total_limit=3, load_best_model_at_end=bool(dev), metric_for_best_model="eval_loss", greater_is_better=False,\n        report_to="none", remove_unused_columns=False, dataloader_num_workers=0, label_names=["labels"],\n        gradient_checkpointing=False,  # 上面已經自己開好\n        seed=42,\n    )\n    print(f"每個 epoch {steps_per_epoch} 步，共 {a.epochs} 個 epoch", flush=True)\n    trainer = Trainer(model=model, args=args, train_dataset=train, eval_dataset=dev or None, data_collator=Collator(tok.pad_token_id))\n    last = trainer_utils.get_last_checkpoint(str(out / "checkpoints")) if (out / "checkpoints").exists() else None\n    if last:\n        print("接續上次的 checkpoint：", last, flush=True)\n    trainer.train(resume_from_checkpoint=last)\n    final = out / "final"\n    trainer.model.save_pretrained(str(final))\n    tok.save_pretrained(str(final))\n    hist = [h for h in trainer.state.log_history]\n    (out / "train_log.json").write_text(json.dumps(hist, ensure_ascii=False, indent=1), encoding="utf-8")\n    (out / "train_args.json").write_text(json.dumps(vars(a), ensure_ascii=False, indent=1), encoding="utf-8")\n    print("完成，adapter 存在", final, "（最佳 checkpoint：", trainer.state.best_model_checkpoint, "）")\n\n\nif __name__ == "__main__":\n    main()\n', 'export_ollama.py': '"""步驟 4（Colab）：把訓練好的 LoRA 變成 Ollama 模型（預設名稱 qwen3.8-27b-typo）。\n\n做法 A（快、省空間）：llama.cpp 把 LoRA 轉成 GGUF adapter → Modelfile「FROM qwen3.8:27b + ADAPTER」\n做法 B（備案）：LoRA 合併回原始權重 → 轉 GGUF → 量化成和官方 qwen3.8:27b 相同的格式 → ollama create\n\n兩種做法建好之後都會做「冒煙測試」：同一批逐字稿，Ollama 版的回覆要和 transformers＋adapter 的回覆大致一致，\n而且要和原版不同（代表 adapter 真的有生效）。不通過就自動改用做法 B。\n產物會複製到 --save（Google Drive），之後每次開新的 Colab 只要 ollama create 就好，不用重訓。\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport re\nimport shutil\nimport subprocess\nimport sys\nfrom pathlib import Path\n\nsys.path.insert(0, str(Path(__file__).resolve().parent))\n\nLLAMA_CPP = Path("/content/llama.cpp")\n\n\ndef sh(cmd: list[str] | str, **kw) -> str:\n    print("$", cmd if isinstance(cmd, str) else " ".join(cmd), flush=True)\n    r = subprocess.run(cmd, shell=isinstance(cmd, str), capture_output=True, text=True, **kw)\n    if r.returncode != 0:\n        print(r.stdout[-3000:], r.stderr[-3000:])\n        raise RuntimeError(f"指令失敗：{cmd}")\n    return r.stdout\n\n\ndef ensure_llama_cpp(build_quantize: bool = False):\n    if not LLAMA_CPP.exists():\n        sh(["git", "clone", "-q", "--depth", "1", "https://github.com/ggml-org/llama.cpp", str(LLAMA_CPP)])\n    sh([sys.executable, "-m", "pip", "install", "-q", str(LLAMA_CPP / "gguf-py")])\n    if build_quantize and not (LLAMA_CPP / "build" / "bin" / "llama-quantize").exists():\n        sh(f"cd {LLAMA_CPP} && cmake -B build -DGGML_CUDA=OFF -DLLAMA_CURL=OFF > /dev/null && "\n           f"cmake --build build --target llama-quantize -j 8 > /dev/null")\n\n\ndef base_quant(base_tag: str) -> str:\n    out = sh(["ollama", "show", base_tag])\n    m = re.search(r"quantization\\s+(\\S+)", out)\n    return m.group(1) if m else "Q4_K_M"\n\n\ndef base_modelfile(base_tag: str) -> str:\n    return sh(["ollama", "show", base_tag, "--modelfile"])\n\n\ndef create(name: str, modelfile: str, workdir: Path):\n    mf = workdir / "Modelfile"\n    mf.write_text(modelfile, encoding="utf-8")\n    sh(["ollama", "create", name, "-f", str(mf)], cwd=str(workdir))\n\n\ndef plan_a(base_hf: str, adapter: str, base_tag: str, name: str, work: Path) -> Path:\n    ensure_llama_cpp()\n    gguf = work / "typo-lora.gguf"\n    sh([sys.executable, str(LLAMA_CPP / "convert_lora_to_gguf.py"), adapter, "--base", base_hf,\n        "--outfile", str(gguf), "--outtype", "f16"])\n    create(name, f"FROM {base_tag}\\nADAPTER {gguf}\\n", work)\n    return gguf\n\n\ndef plan_b(base_hf: str, adapter: str, base_tag: str, name: str, work: Path) -> Path:\n    import torch\n    import lm_utils as L\n    ensure_llama_cpp(build_quantize=True)\n    merged = work / "merged"\n    if not (merged / "config.json").exists():\n        model = L.load_model(base_hf, adapter)\n        model = model.merge_and_unload()\n        model.save_pretrained(str(merged), max_shard_size="5GB")\n        del model\n        torch.cuda.empty_cache()\n        for f in Path(base_hf).iterdir():   # tokenizer、chat template、processor 設定\n            if f.suffix in (".json", ".jinja", ".txt") and not f.name.startswith("model") and f.name != "config.json":\n                shutil.copy(f, merged / f.name)\n    q8 = work / "typo-q8_0.gguf"\n    if not q8.exists():\n        sh([sys.executable, str(LLAMA_CPP / "convert_hf_to_gguf.py"), str(merged), "--outfile", str(q8), "--outtype", "q8_0"])\n    shutil.rmtree(merged, ignore_errors=True)\n    qtype = base_quant(base_tag)\n    final = work / f"typo-{qtype}.gguf"\n    if qtype.upper() in ("Q8_0",):\n        q8.rename(final)\n    else:\n        sh([str(LLAMA_CPP / "build" / "bin" / "llama-quantize"), "--allow-requantize", str(q8), str(final), qtype])\n        q8.unlink(missing_ok=True)\n    mf = base_modelfile(base_tag)\n    mf = re.sub(r"^FROM .*$", f"FROM {final}", mf, count=1, flags=re.M)\n    mf = re.sub(r"^ADAPTER .*$\\n?", "", mf, flags=re.M)\n    create(name, mf, work)\n    return final\n\n\ndef smoke(data: str, base_tag: str, name: str, base_hf: str, adapter: str, n: int = 4) -> bool:\n    """Ollama 版要「跟原版不同」而且「跟 transformers+adapter 的結果夠像」。"""\n    import difflib\n    import eval_typo as E\n    rows = [json.loads(x) for x in Path(data, "sft_dev.jsonl").open(encoding="utf-8")][:n]\n    try:   # 先跑 transformers（此時 Ollama 還沒載入模型，顯示卡記憶體夠）\n        import gc\n        import torch\n        b = E.HFBackend(base_hf, adapter)\n        hf = b.generate(rows)\n        del b\n        gc.collect()\n        torch.cuda.empty_cache()\n    except Exception as exc:\n        print("（transformers 比對略過：", exc, "）")\n        hf = [r["target"] for r in rows]\n    o_base = E.OllamaBackend(base_tag, "http://127.0.0.1:11434").generate(rows)\n    o_ft = E.OllamaBackend(name, "http://127.0.0.1:11434").generate(rows)\n    diff_base = sum(a.strip() != b.strip() for a, b in zip(o_base, o_ft))\n    sim = sum(difflib.SequenceMatcher(None, a, b).ratio() for a, b in zip(o_ft, hf)) / len(rows)\n    for r, a, b, c in zip(rows, o_base, o_ft, hf):\n        print("--", r["id"], "\\n  原版:", a[:160].replace("\\n", " / "), "\\n  微調:", b[:160].replace("\\n", " / "),\n              "\\n  HF  :", c[:160].replace("\\n", " / "))\n    print(f"冒煙測試：{diff_base}/{len(rows)} 批和原版不同；和 transformers 版相似度 {sim:.2f}")\n    return diff_base > 0 and sim >= 0.6\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--base-hf", required=True)\n    ap.add_argument("--adapter", required=True)\n    ap.add_argument("--data", required=True)\n    ap.add_argument("--base-tag", default="qwen3.8:27b")\n    ap.add_argument("--name", default="qwen3.8-27b-typo")\n    ap.add_argument("--work", default="/content/ft_export")\n    ap.add_argument("--save", required=True, help="Google Drive 上存放產物的資料夾")\n    ap.add_argument("--plan", choices=["auto", "a", "b"], default="auto")\n    a = ap.parse_args()\n    work = Path(a.work)\n    work.mkdir(parents=True, exist_ok=True)\n    save = Path(a.save)\n    save.mkdir(parents=True, exist_ok=True)\n\n    used, artifact = None, None\n    if a.plan in ("auto", "a"):\n        try:\n            artifact = plan_a(a.base_hf, a.adapter, a.base_tag, a.name, work)\n            if smoke(a.data, a.base_tag, a.name, a.base_hf, a.adapter):\n                used = "a"\n            else:\n                print("做法 A 的冒煙測試沒通過")\n        except Exception as exc:\n            print("做法 A 失敗：", exc)\n    if used is None and a.plan in ("auto", "b"):\n        artifact = plan_b(a.base_hf, a.adapter, a.base_tag, a.name, work)\n        if not smoke(a.data, a.base_tag, a.name, a.base_hf, a.adapter):\n            raise SystemExit("做法 B 冒煙測試也沒通過，請檢查訓練結果")\n        used = "b"\n    if used is None:\n        raise SystemExit("匯出失敗")\n    shutil.copy(artifact, save / artifact.name)\n    info = {"plan": used, "file": artifact.name, "name": a.name, "base_tag": a.base_tag,\n            "modelfile": (work / "Modelfile").read_text(encoding="utf-8").replace(str(artifact), artifact.name)}\n    (save / "ollama_model.json").write_text(json.dumps(info, ensure_ascii=False, indent=1), encoding="utf-8")\n    print("完成：", info)\n\n\nif __name__ == "__main__":\n    main()\n', 'compare.py': '"""步驟 5：原版 vs 微調版，產生比較報告與「要不要換上去」的判定。\n\n判定規則（每個語言各自判斷）：\n  1. F0.5 比原版高（F0.5 偏重準確率：寧可少抓，不要亂抓，和 prompt 的要求一致）\n  2. 準確率（precision）沒有比原版低超過 2 個百分點\n  3. 套用模型建議後的 CER/WER 不比原版差\n三項都過才建議換上微調版。\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport json\nfrom pathlib import Path\n\nLABEL = {"zh": "國語", "en": "英文"}\n\n\ndef pct(x):\n    return "—" if x is None else f"{x * 100:.1f}%"\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--results", required=True)\n    ap.add_argument("--base", default="base_ollama")\n    ap.add_argument("--ft", default="ft_ollama")\n    ap.add_argument("--out", required=True)\n    a = ap.parse_args()\n    R = Path(a.results)\n    base = json.loads((R / f"{a.base}.json").read_text(encoding="utf-8"))\n    ft = json.loads((R / f"{a.ft}.json").read_text(encoding="utf-8"))\n    lines = ["# 錯字校正微調結果", "",\n             f"- 原版：`{base[\'_meta\'][\'model\']}`\u3000微調版：`{ft[\'_meta\'][\'model\']}`",\n             "- 測試資料：沒有參與訓練的會議（國語 AISHELL-4 test、英文 AMI test），只用真實 ASR 錯誤，不含人工注入的錯字",\n             "- 一筆建議算「對」= 套用後和人工逐字稿一致；評分用的是專案正式的解析程式", ""]\n    verdict = {}\n    for lang in ["zh", "en"]:\n        if lang not in base or lang not in ft:\n            continue\n        b, f = base[lang], ft[lang]\n        m = b.get("error_rate_metric", "CER")\n        lines += [f"## {LABEL[lang]}（{b[\'chunks\']} 批、標準答案 {b[\'gold\']} 處）", "",\n                  "| 指標 | 原版 | 微調版 |", "|---|---|---|",\n                  f"| 準確率（建議中有幾成是對的） | {pct(b[\'precision\'])} | {pct(f[\'precision\'])} |",\n                  f"| 召回率（錯字被抓到幾成） | {pct(b[\'recall\'])} | {pct(f[\'recall\'])} |",\n                  f"| F0.5（綜合，偏重準確率） | {b[\'f0.5\']:.3f} | {f[\'f0.5\']:.3f} |",\n                  f"| 建議總數 / 正確數 | {b[\'pred\']} / {b[\'tp\']} | {f[\'pred\']} / {f[\'tp\']} |",\n                  f"| 沒有錯字的批次被亂報的比例 | {pct(b[\'false_alarm_rate_on_clean_chunks\'])} | {pct(f[\'false_alarm_rate_on_clean_chunks\'])} |",\n                  f"| {m}：原始 ASR | {pct(b[\'error_rate\'][\'before\'])} | {pct(f[\'error_rate\'][\'before\'])} |",\n                  f"| {m}：全部套用模型建議後 | {pct(b[\'error_rate\'][\'after_model\'])} | {pct(f[\'error_rate\'][\'after_model\'])} |",\n                  f"| {m}：全部套用標準答案（上限） | {pct(b[\'error_rate\'][\'after_gold\'])} | {pct(f[\'error_rate\'][\'after_gold\'])} |", ""]\n        ok1 = f["f0.5"] > b["f0.5"]\n        ok2 = f["precision"] >= b["precision"] - 0.02\n        ok3 = f["error_rate"]["after_model"] <= b["error_rate"]["after_model"]\n        verdict[lang] = ok1 and ok2 and ok3\n        lines += [f"判定：{\'✅ 建議換上微調版\' if verdict[lang] else \'❌ 維持原版\'}"\n                  f"（F0.5 提升 {\'是\' if ok1 else \'否\'}、準確率沒下降 {\'是\' if ok2 else \'否\'}、{m} 沒變差 {\'是\' if ok3 else \'否\'}）", ""]\n    (Path(a.out)).write_text("\\n".join(lines), encoding="utf-8")\n    (R / "verdict.json").write_text(json.dumps(verdict, ensure_ascii=False), encoding="utf-8")\n    print("\\n".join(lines))\n\n\nif __name__ == "__main__":\n    main()\n'}
for name, src in EMBED.items():
    open(f'{FT}/{name}', 'w', encoding='utf-8').write(src)
print('微調程式：', ', '.join(EMBED))

結果資料夾： /content/drive/MyDrive/MeetSum_finetune
name, memory.total [MiB]
NVIDIA A100-SXM4-80GB, 81920 MiB

磁碟剩餘 107 GB
微調程式： ft_common.py, lm_utils.py, build_asr_data.py, build_sft.py, eval_typo.py, train_lora.py, export_ollama.py, compare.py


## 1. 準備專案與正式環境（Ollama＋qwen3.8:27b＋ASR 模型）
沿用 `run_all.py`，和平常 demo 的環境完全相同。

In [22]:
NAMES = ('Taiwanese_Meeting_Assistant', 'test2')
REPO = 'https://github.com/henry19990130/Taiwanese_Meeting_Assistant.git'
cands = []
for n in NAMES if os.path.isdir('/content/drive') else []:
    pats = ['/content/drive/MyDrive/' + '*/' * d + n + '/run_all.py' for d in range(0, 3)]
    pats += ['/content/drive/Othercomputers/' + '*/' * d + n + '/run_all.py' for d in range(1, 5)]
    cands += [p for pat in pats for p in glob.glob(pat)]
if cands:
    RUN_ALL_SOURCE = os.path.dirname(max(set(cands), key=os.path.getmtime))
else:
    RUN_ALL_SOURCE = '/content/_src_github'
    if not os.path.isdir(RUN_ALL_SOURCE):
        subprocess.run(['git', 'clone', '-q', '--depth', '1', REPO, RUN_ALL_SOURCE], check=True)
print('專案來源：', RUN_ALL_SOURCE)
RUN_STEPS = {'sync', 'setup'}
exec(open(os.path.join(RUN_ALL_SOURCE, 'run_all.py'), encoding='utf-8').read())
PROJECT = '/content/Taiwanese_Meeting_Assistant'
os.environ['PROJECT_BACKEND'] = f'{PROJECT}/backend'
assert os.path.exists(f'{PROJECT}/backend/app/services/typo_service.py'), '專案程式沒有複製成功'
run([sys.executable, '-m', 'pip', 'install', '-q', 'pypinyin', 'jieba', 'jellyfish', 'opencc-python-reimplemented', 'librosa'])

專案來源： /content/drive/Othercomputers/我的 PC/Taiwanese_Meeting_Assistant

① 找到專案、複製最新程式　（已執行 0 秒）
ℹ️ 沒有 HF_TOKEN：中文／台語／英文可用；客語模型需要先在 .env 加 HF_TOKEN=... 才能下載
   （VS Code 的 Colab 擴充讀不到 Colab 金鑰，請寫在 .env）
✅ 專案來源： /content/drive/Othercomputers/我的 PC/Taiwanese_Meeting_Assistant
✅ 執行位置： /content/Taiwanese_Meeting_Assistant

② 安裝環境　（已執行 11 秒）
✅ 已經安裝過，而且資料庫、Ollama 都在執行 → 跳過安裝

完成　（已執行 11 秒）
套用修正版： ft_common.py
套用修正版： compare.py
套用修正版： eval_typo.py
套用修正版： build_asr_data.py
套用修正版： train_lora.py
套用修正版： export_ollama.py
$ /usr/bin/python3 -m pip install -q pypinyin jieba jellyfish opencc-python-reimplemented librosa


## 2. 產生資料：公開會議錄音 → 專案 ASR 辨識 → 和人工逐字稿比對
約 40–70 分鐘。中斷後重跑會從還沒做的會議繼續。

In [23]:
if not done(f'{DATA}/.asr_done'):
    run([sys.executable, f'{FT}/build_asr_data.py', '--out', DATA, '--scale', str(SCALE)], 'build_asr')
    open(f'{DATA}/.asr_done', 'w').close()
run([sys.executable, f'{FT}/build_sft.py', '--data', DATA], 'build_sft')
print(open(f'{DATA}/stats.json', encoding='utf-8').read())

套用修正版： ft_common.py
套用修正版： compare.py
套用修正版： eval_typo.py
套用修正版： build_asr_data.py
套用修正版： train_lora.py
套用修正版： export_ollama.py
$ /usr/bin/python3 /content/ft_typo/build_sft.py --data /content/drive/MyDrive/MeetSum_finetune/data2


/usr/local/lib/python3.13/dist-packages/jieba/_compat.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Building prefix dict from the default dictionary ...
Loading model from cache /tmp/jieba.cache
Loading model cost 0.925 seconds.
Prefix dict has been built successfully.
train {"zh": {"examples": 422, "clean_examples": 17, "edits": 3259, "by_source": {"asr": 2790, "synthetic": 469}, "kinds": {"real": 211, "aug": 211}}, "en": {"examples": 458, "clean_examples": 164, "edits": 619, "by_source": {"asr": 348, "synthetic": 271}, "kinds": {"real": 229, "aug": 229}}}
dev {"zh": {"examples": 9, "clean_examples": 0, "edits": 62, "by_source": {"asr": 62}, "kinds": {"real": 9}}, "en": {"examples": 16, "clean_examples": 12, "edits": 5, "by_source": {"asr": 5}, "kinds": {"real": 

## 3. 原版 qwen3.8:27b 的基準分數（走正式系統的 Ollama 呼叫）

In [24]:
if not done(f'{WORK}/results/base_ollama.json'):
    run([sys.executable, f'{FT}/eval_typo.py', '--data', DATA, '--backend', 'ollama', '--model', BASE_TAG,
         '--name', 'base_ollama', '--out', f'{WORK}/results'], 'eval_base')
r = json.load(open(f'{WORK}/results/base_ollama.json', encoding='utf-8'))
for k in ['zh', 'en']:
    if k in r: print(k, {x: r[k][x] for x in ['precision', 'recall', 'f0.5', 'false_alarm_rate_on_clean_chunks', 'error_rate']})

zh {'precision': 0.1092, 'recall': 0.1889, 'f0.5': 0.1193, 'false_alarm_rate_on_clean_chunks': 0.0, 'error_rate': {'before': 0.2478, 'after_model': 0.2421, 'after_gold': 0.2423}}
en {'precision': 0.0, 'recall': 0.0, 'f0.5': 0.0, 'false_alarm_rate_on_clean_chunks': 0.5882, 'error_rate': {'before': 0.198, 'after_model': 0.1984, 'after_gold': 0.1973}}


## 4. 下載原始權重並訓練 LoRA
訓練前先讓 Ollama 釋放顯示卡記憶體。checkpoint 存在雲端硬碟，斷線後重跑會接續。約 3–5 小時。

In [25]:
subprocess.run('ollama ps | tail -n +2 | awk \'{print $1}\' | xargs -r -n1 ollama stop', shell=True)
run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'transformers', 'peft', 'accelerate', 'bitsandbytes'], 'pip_train')
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'flash-linear-attention', 'causal-conv1d'], capture_output=True, text=True)
print('線性注意力加速套件：', '已安裝' if r.returncode == 0 else '安裝失敗，改用較慢的版本（結果相同）')
if not done(f'{WORK}/run1/final/adapter_model.safetensors'):
    if not done(f'{BASE_HF_DIR}/model.safetensors.index.json'):
        from huggingface_hub import snapshot_download
        snapshot_download(BASE_HF, local_dir=BASE_HF_DIR, max_workers=16)
    args = [sys.executable, f'{FT}/train_lora.py', '--base', BASE_HF_DIR, '--data', DATA, '--out', f'{WORK}/run1']
    if done(f'{WORK}/run1/.use_4bit'):
        args.append('--load-4bit')
    try:
        run(args, 'train')
    except RuntimeError:
        log = open(f'{WORK}/logs/train.log', encoding='utf-8').read()[-5000:]
        if 'out of memory' in log.lower() and '--load-4bit' not in args:
            print('⚠️ 顯示卡記憶體不夠（', GPU_GB, 'GB ）→ 改用 QLoRA（4-bit 載入原始權重）重新訓練')
            open(f'{WORK}/run1/.use_4bit', 'w').close()
            shutil.rmtree(f'{WORK}/run1/checkpoints', ignore_errors=True)
            run(args + ['--load-4bit'], 'train')
        else:
            raise
print(open(f'{WORK}/run1/train_args.json').read())

套用修正版： ft_common.py
套用修正版： compare.py
套用修正版： eval_typo.py
套用修正版： build_asr_data.py
套用修正版： train_lora.py
套用修正版： export_ollama.py
$ /usr/bin/python3 -m pip install -q -U transformers peft accelerate bitsandbytes
線性注意力加速套件： 已安裝
{
 "base": "/content/hf/Qwen3.8-27B",
 "data": "/content/drive/MyDrive/MeetSum_finetune/data2",
 "out": "/content/drive/MyDrive/MeetSum_finetune/run1",
 "epochs": 2.0,
 "lr": 0.0001,
 "rank": 16,
 "alpha": 32,
 "max_len": 6144,
 "grad_accum": 16,
 "save_steps": 20,
 "dev_limit": 48,
 "load_4bit": false,
 "max_steps": -1,
 "max_train": 1200
}


## 5. 轉成 Ollama 模型 `qwen3.8-27b-typo`（原版不動）

In [26]:
if not done(f'{WORK}/export/ollama_model.json'):
    if not done(f'{BASE_HF_DIR}/model.safetensors.index.json'):
        from huggingface_hub import snapshot_download
        snapshot_download(BASE_HF, local_dir=BASE_HF_DIR, max_workers=16)
    run([sys.executable, f'{FT}/export_ollama.py', '--base-hf', BASE_HF_DIR, '--adapter', f'{WORK}/run1/final',
         '--data', DATA, '--base-tag', BASE_TAG, '--name', FT_TAG, '--save', f'{WORK}/export'], 'export')
else:
    info = json.load(open(f'{WORK}/export/ollama_model.json', encoding='utf-8'))
    open(f'{WORK}/export/Modelfile', 'w').write(info['modelfile'])
    subprocess.run(['ollama', 'create', FT_TAG, '-f', 'Modelfile'], cwd=f'{WORK}/export', check=True)
print(subprocess.run(['ollama', 'list'], capture_output=True, text=True).stdout)

套用修正版： ft_common.py
套用修正版： compare.py
套用修正版： eval_typo.py
套用修正版： build_asr_data.py
套用修正版： train_lora.py
套用修正版： export_ollama.py
$ /usr/bin/python3 /content/ft_typo/export_ollama.py --base-hf /content/hf/Qwen3.8-27B --adapter /content/drive/MyDrive/MeetSum_finetune/run1/final --data /content/drive/MyDrive/MeetSum_finetune/data2 --base-tag qwen3.8:27b --name qwen3.8-27b-typo --save /content/drive/MyDrive/MeetSum_finetune/export
Ollama 診斷資料已存： /content/drive/MyDrive/MeetSum_finetune/export/ollama_diagnose.txt
transformers A/B 評測：56 批 × 2（原版／微調版）

== base_hf（不掛 LoRA）0%|██████████| 1184/1184 [01:05<00:00, 17.98it/s]]
[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slow

## 6. 微調版的分數，和原版比較

In [28]:
if not done(f'{WORK}/results/ft_ollama.json'):
    run([sys.executable, f'{FT}/eval_typo.py', '--data', DATA, '--backend', 'ollama', '--model', FT_TAG,
         '--name', 'ft_ollama', '--out', f'{WORK}/results'], 'eval_ft')
run([sys.executable, f'{FT}/compare.py', '--results', f'{WORK}/results', '--out', f'{WORK}/results/report.md'], 'compare')
print(open(f'{WORK}/results/report.md', encoding='utf-8').read())

套用修正版： ft_common.py
套用修正版： compare.py
套用修正版： eval_typo.py
套用修正版： build_asr_data.py
套用修正版： train_lora.py
套用修正版： export_ollama.py
$ /usr/bin/python3 /content/ft_typo/compare.py --results /content/drive/MyDrive/MeetSum_finetune/results --out /content/drive/MyDrive/MeetSum_finetune/results/report.md
# 錯字校正微調結果

- 原版：`Qwen3.8-27B bf16`　微調版：`Qwen3.8-27B bf16 + LoRA`
- 測試資料：沒有參與訓練的會議（國語 AISHELL-4 test、英文 AMI test），只用真實 ASR 錯誤，不含人工注入的錯字
- 一筆建議算「對」= 套用後和人工逐字稿一致；評分用的是專案正式的解析程式
- 比較方式：同一份 Qwen3.8-27B（bf16）權重，掛 LoRA vs 不掛 LoRA，其餘完全相同
- 參考：正式系統的 Ollama 版 qwen3.8:27b（4-bit 量化）在同一份測試上的結果：國語 準確率 10.9%、寬鬆改對 57.8%、改錯 6.2%；英文 準確率 0.0%、寬鬆改對 24.1%、改錯 44.8%

## 國語（31 批、標準答案 270 處）

| 指標 | 原版 | 微調版 |
|---|---|---|
| 準確率（建議中有幾成是對的） | 10.0% | 13.4% |
| 召回率（錯字被抓到幾成） | 18.9% | 22.6% |
| F0.5（綜合，偏重準確率） | 0.110 | 0.146 |
| 建議總數 / 正確數 | 512 / 51 | 455 / 61 |
| 寬鬆：建議讓文字更接近人工逐字稿的比例 | 56.0% | 51.2% |
| 寬鬆：建議反而改錯的比例 | 5.7% | 15.8% |
| 沒有錯字的批次被亂報的比例 | 0.0% | 0.0% |
| CER：原始 ASR | 24.8% | 24.8% |
| CER：全部套用模型建議後 | 24.2% 

OSError: [Errno 107] Transport endpoint is not connected